In [1]:
import torch
import torch.nn as nn
import torchvision
from torchvision import transforms

In [2]:
train_dataset = torchvision.datasets.ImageFolder(
    "datasets/butterflies",
    transform=transforms.Compose(
        [
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize([0.5], [0.5]),
        ]
    ),
)

In [3]:
class PatchEmbed(nn.Module):
    """ """

    def __init__(
        self,
        img_size: int | tuple[int, int] = 224,
        patch_size: int | tuple[int, int] = 16,
        in_chans: int = 3,
        embed_dim: int = 768,
        bias: bool = True,
    ):
        super().__init__()

        if isinstance(img_size, int):
            img_size = (img_size, img_size)
        if isinstance(patch_size, int):
            patch_size = (patch_size, patch_size)

        self.img_size = img_size
        self.patch_size = patch_size
        self.grid_size = tuple(
            [
                s // p
                for s, p in zip(
                    img_size,
                    patch_size,
                )
            ]
        )

        self.num_patches = self.grid_size[0] * self.grid_size[1]
        self.proj = nn.Conv2d(
            in_chans,
            embed_dim,
            kernel_size=patch_size,
            stride=patch_size,
            bias=bias,
        )

    def forward(self, x: torch.Tensor):
        # (B, C, H, W) <= x

        # B: batch
        # C: channel
        # H: height (original)
        # W: width (original)

        # D: embed_dim
        # h: height (projected)
        # w: width (projected)
        # L: seqlen = h*w

        # (B, D, h, w)
        x = self.proj(x)

        # (B, D, L)
        # (B, L, D)
        x = x.flatten(2).transpose(1, 2)

        return x


In [4]:
patch_embed = PatchEmbed()

img_tensor, label = train_dataset[0]
image_batch = img_tensor[None, :]
print(image_batch.shape)
image_projected = patch_embed(image_batch)
print(image_projected.shape)
print(patch_embed.grid_size)
print(patch_embed.num_patches)

torch.Size([1, 3, 224, 224])
torch.Size([1, 196, 768])
(14, 14)
196
